In [ ]:
import requests
import urllib.error, urllib.parse
from urllib.parse import urljoin
from google.colab import drive
from PIL import Image
from io import BytesIO
import os
from bs4 import BeautifulSoup
import re

# Google Driveのマウントポイント
drive_root_dir ='/content/drive'
drive.mount(drive_root_dir)

# Google Driveの中のフォルダ名
save_folder = '/My Drive/scraping/'

# ダウンロード元のURL
url = 'https://docs.crssrds.jp/learn/target-data/'

# ファイル名を抽出するための正規表現のパターン
pattern1 = ".+(jpg|png|bmp)$"
pattern2 = "^http"

# 保存先のフォルダの場所をマウントポイント + フォルダ名で作成。
target_folder = drive_root_dir + save_folder

# そのフォルダがあるかどうかチェック
if not os.path.exists(target_folder):
  # なければ作成する
  os.mkdir(target_folder)

try:
    # 対象のサイトにアクセスしてサイトのデータを取得する。
    html = urllib.request.urlopen(url)

    # BeautifulSoupを使って中のデータにアクセスできるようにする
    soup = BeautifulSoup(html, "html.parser")

    # imgタグをすべて探す
    images = soup.find_all('img')

    # 見つけたimgタグを表示する
    print('このページの中の画像(imgタグ)は以下の通りです')
    for img in images:
      print(img)
    print('----------------')

    # imgタグをひとつずつ取り出して処理
    for img in images:
      if not img is None:
        # img タグのsrcからURLを取得
        link = img.get('src')
        print('urlは {link} です'.format(link=link))
        if re.search(pattern2, link, re.IGNORECASE) is None:
          # URLがhttpsで始まっていなければ、URLを接続して新しいURLを作る
          print('{url} と {link} をつないで'.format(url=url, link=link))
          link = urljoin(url,link)
          print('{link} にする'.format(link=link))
        if not re.search(pattern1, link, re.IGNORECASE) is None:
          # ファイル名の最後が jpg, png, bmpだったらダウンロード処理を行う。
          # URLを"/"で区切ってバラバラにする
          list = link.split("/")
          for item in list:
            print(item, end=', ')
          # バラバラにした最後のひとつを取る（それがファイル名）
          filename = list[len(list)-1]

          # 画像のURLにアクセスしてデータを取得する
          response = requests.get(link)
          # 結果のコードをチェック
          print(response.status_code)
          # 読み込んだデータを画像に変換
          img = Image.open(BytesIO(response.content))

          # 画像を保存先フォルダに保存
          img.save(target_folder + filename)
          print('ダウンロード完了：',filename)
          print('----------------')
except Exception as e:
  # エラーが発生したらエラーコードを表示
  print('エラーが発生しました')
  print(e)